# 1. Embeddings in PyTorch 

For this we gonna use the 

```python

embedding = nn.Embedding()

```

---

## 1.1 Args For Embedding



```python

nn.Embedding(
    num_embeddings = X,      # (int) Vocab Size
    embedding_dim,       # (int) Dimension of each embedding
    padding_idx=None,    # (int) Index to ignore (complete batchs with differents sizes of tokens)
    max_norm=None,       # (float) Normalize per max_norm, limits the max value 
    norm_type=2.0,       # (float) Type of norm (L2=2.0, L1=1.0)
)

```

---

## 1.2 How It Works Internally

When you create an embedding:

```python
embed = nn.Embedding(vocab_size=50, embedding_dim=768)
```

PyTorch creates a **random matrix (50, 768)**:

```
Row 0  (ID 0)  → [-0.2,  0.5,  0.1, ..., 0.4]  (768 dimensions)
Row 1  (ID 1)  → [ 0.8, -0.3,  0.6, ..., 0.2]
Row 2  (ID 2)  → [ 0.1,  0.2, -0.4, ..., -0.5]
Row 3  (ID 3)  → [-0.5,  0.9,  0.2, ..., 0.7]
...
Row 29 (ID 29) → [ 0.1, 0.2, -0.4, ..., 0.3]
...
Row 49 (ID 49) → [-0.1, 0.3,  0.5, ..., -0.2]
```

**Position in the matrix = Token ID**


### Practical Example


```python
import torch
import torch.nn as nn

# Create embedding: 5 words → 3 dimensions
embed = nn.Embedding(num_embeddings=5, embedding_dim=3)

print(embed.weight)
# Internal matrix (5, 3):
# tensor([[-0.2,  0.5,  0.1],
#         [ 0.8, -0.3,  0.6],
#         [ 0.1,  0.2, -0.4],
#         [-0.5,  0.9,  0.2],
#         [ 0.3, -0.1,  0.7]])

# Pass token IDs
tokens = torch.tensor([1, 3, 0])

# Lookup in table
output = embed(tokens)

# Input (3, )
# Result (3, 3):
# tensor([[ 0.8, -0.3,  0.6],   ← row 1
#         [-0.5,  0.9,  0.2],   ← row 3
#         [-0.2,  0.5,  0.1]])  ← row 0
```


### Manual Equivalent

```python
# Create embedding manually:
embedding_matrix = torch.randn(5, 3)
tokens = torch.tensor([1, 3, 0])

# Do lookup:
output = embedding_matrix[tokens]  # Simple indexing!

# Same thing nn.Embedding does!
```
---

# 2. Types


## 2.1 Absolute Positional Embedding

Embedding + Positional Embedding

```python

# 1. Create the matrix of embedding and pos embedding
embedding = nn.Embedding(vocab_size, d_model)
pos_embedding = nn.Embedding(max_seq_len, d_model)

# 2. Get the values of the shape
b, t = x.shape # Get the shape of the X

# 3. Create the vector for the positional embedding
vector_size = torch.arange(t, device = x.device) # Create a vector with the size of the actual number of tokens and ** PUT IN THE SAME DEVICE AS X (GPU) **

# 4. Embedding and sum to get the final result
embedding_final = embedding(x) + pos_embedding(vector_size)


```


## 2.2 RoPE

Rotation the dimensions of the tokens in pairs

```python

def apply_rope(embedding, positional, d_model, base = 10000) -> torch.Tensor:

    for i in range (0, d_model, 2): 
        '''
        An loop from 0 to d_model, in step of 2

        first step = 0
        second step = 2
        third step = 4

        '''
        theta = positional * (base ** (- 2 * i / d_model)) # Calculates the theta, i represents the index of the pair

        e0 = embedding[i] # Get the value of e0 (X) of the pair
        e1 = embedding[i + 1] # Get the value of e1 (y) of the pair

        cos_theta = np.cos(theta) # Get the cos0
        sin_theta = np.sin(theta) # Get the sin0


        embedding[i] = (cos_theta * e0) - (sin_theta * e1) # Rotate the e0 (X) of the pair 
        embedding[i + 1] = (sin_theta * e0) + (cos_theta * e1) # Rotate the e1 (Y)d of the pair

    return embedding


def forward(self, x):
    x.shape = (B, T, D_MODEL)
    # B=2, T=5, D_MODEL=10

    for b, sequence in enumerate(x):
    # b = index of the batch (0 or 1)
    # sequence = this sequence (shape: T, D_MODEL)
    
        for positional, embedding in enumerate(sequence):
            # positional = index of the token (0, 1, 2, 3, 4)
            # embedding = that token (shape: D_MODEL)
            embedding = apply_rope(embedding, positional, d_model)


```

## 2.3 Alibibs

to do yet